In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("ClickstreamPipeline") \
    .getOrCreate()

In [18]:
schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device", StringType(), True),
    StructField("country", StringType(), True),
    StructField("event_date", StringType(), True),
    StructField("session_time_sec", IntegerType(), True)
])

In [19]:
df = spark.read.csv(
    "events.csv",
    header=True,
    schema=schema
)

In [20]:
df

DataFrame[event_id: int, user_id: string, event_type: string, device: string, country: string, event_date: string, session_time_sec: int]

In [21]:
null_user = df.filter(F.col("user_id").isNull()).count()
null_event = df.filter(F.col("event_type").isNull()).count()

print("NULL user_id:", null_user)
print("NULL event_type:", null_event)

NULL user_id: 0
NULL event_type: 0


In [22]:
bad_df = df.filter(
    F.col("event_id").isNull() |
    F.col("event_type").isNull()
)

In [23]:
good_df = df.filter(
    F.col("event_id").isNotNull() &
    F.col("event_type").isNotNull()
)

In [24]:
good_df = good_df.fillna({
    "user_id": "anonymous"
})

In [25]:
users_data = [
    ("101", "Neel", "Free"),
    ("102", "John", "Premium"),
    ("103", "Priya", "Free")
]

users_df = spark.createDataFrame(
    users_data,
    ["user_id", "user_name", "subscription_type"]
)

In [26]:
joined_df = good_df.join(
    users_df,
    on="user_id",
    how="left"
)

In [27]:
joined_df

DataFrame[user_id: string, event_id: int, event_type: string, device: string, country: string, event_date: string, session_time_sec: int, user_name: string, subscription_type: string]

In [28]:
joined_df = joined_df.withColumn(
    "session_minutes",
    F.col("session_time_sec") / 60
)

In [29]:
joined_df = joined_df.withColumn(
    "session_minutes",
    F.col("session_time_sec") / 60
)

In [30]:
final_df = joined_df.groupBy(
    "user_id",
    "country",
    "event_date"
).agg(
    F.count("event_id").alias("total_events"),
    F.sum("session_minutes").alias("total_session_minutes"),
    F.collect_set("device").alias("devices_used")
)

In [31]:
final_df

DataFrame[user_id: string, country: string, event_date: string, total_events: bigint, total_session_minutes: double, devices_used: array<string>]

In [34]:
final_df.write \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .parquet("./Neelusree")

In [36]:
final_df

DataFrame[user_id: string, country: string, event_date: string, total_events: bigint, total_session_minutes: double, devices_used: array<string>]

In [37]:
bad_df.write \
    .mode("overwrite") \
    .parquet("./Sree")